In [1]:
import re, time, csv
from datasets import load_dataset
from huggingface_hub import hf_hub_download

In [2]:
def is_var(x):
    return isinstance(x, str) and len(x) == 1 and x.isupper()

def walk(x, subst):
    while is_var(x) and x in subst:
        x = subst[x]
    return x

def apply(atom, subst):
    p, args = atom
    return (p, tuple(walk(a, subst) for a in args))

In [3]:
def match(a, b, subst):
    p1, x1 = a
    p2, x2 = b
    if p1 != p2 or len(x1) != len(x2):
        return None
    subst = dict(subst)
    for t1, t2 in zip(x1, x2):
        t1, t2 = walk(t1, subst), walk(t2, subst)
        if t1 == t2:
            continue
        if is_var(t1):
            subst[t1] = t2; continue
        if is_var(t2):
            subst[t2] = t1; continue
        return None
    return subst

In [4]:
def forward(facts, rules, blocked=None):
    blocked = blocked or set()
    facts = set(facts) - blocked
    for _ in range(300):
        new = set()
        for body, head in rules:
            if head in blocked:
                continue
            substs = [{}]
            for b in body:
                nxt = []
                for s in substs:
                    for f in facts:
                        m = match(b, f, s)
                        if m is not None:
                            nxt.append(m)
                substs = nxt
                if not substs:
                    break
            for s in substs:
                h = apply(head, s)
                if h not in facts and h not in blocked:
                    new.add(h)
        if not new:
            break
        facts |= new
    return facts

In [5]:
def norm(text):
    text = text.strip().lower().rstrip(".,;:")
    for w in ("the ", "a ", "an "):
        if text.startswith(w):
            text = text[len(w):]
    if text.startswith("not "):
        return "Not" + norm(text[4:])
    return "".join(w.capitalize() for w in re.findall(r"[a-z0-9]+", text))

def name(text):
    text = text.strip().lower()
    for w in ("the ", "a ", "an "):
        if text.startswith(w):
            text = text[len(w):]
    return text.capitalize() if text else text

def parse_verb(text):
    text = text.strip().rstrip(".")
    m = re.match(r"^(?:the\s+|a\s+|an\s+)?([a-z]+)\s+"r"(?:(does not|doesn't|do not|don't)\s+)?"r"([a-z]+)\s+"r"(?:the\s+|a\s+|an\s+)?([a-z]+)$",text, re.I,)
    if not m:
        return None
    subj, neg, verb, obj = m.group(1), m.group(2), m.group(3), m.group(4)
    if verb.lower() in ("is", "are", "was", "were", "has", "have"):
        return None
    if verb.lower().endswith("s"):
        verb = verb[:-1]
    p = norm(verb)
    if neg:
        p = "Not" + p
    return (p, (name(subj), name(obj)))

def parse_atom(text):
    text = text.strip().rstrip(".")
    m = re.match(r"^(they|he|she|it|someone|somebody|something)\s+(?:is|are)\s+(.+)$", text, re.I)
    if m:
        return (norm(m.group(2)), ("X",))
    m = re.match(r"^(.+?)\s+is\s+not\s+(.+)$", text, re.I)
    if m:
        return ("Not" + norm(m.group(2)), (name(m.group(1)),))
    m = re.match(r"^(.+?)\s+is\s+(.+)$", text, re.I)
    if m:
        return (norm(m.group(2)), (name(m.group(1)),))
    v = parse_verb(text)
    if v:
        return v
    return (norm(text), ("X",))

def split_and(text):
    parts = []
    for c in re.split(r",\s*", text):
        for s in re.split(r"\s+and\s+", c, flags=re.I):
            if s.strip():
                parts.append(s.strip())
    return parts

def parse_condition(text):
    return [parse_atom(p) for p in split_and(text)]

def parse_sentence(s):
    s = s.strip()
    if not s:
        return [], []
    m = re.match(r"^We know that (.+?)\.?$", s, re.I)
    if m:
        facts, rules = [], []
        for piece in re.split(r",\s*and\s+|,\s+", m.group(1)):
            piece = piece.strip()
            m2 = re.match(r"^(.+?)\s+together\s+cause\s+(.+?)$", piece, re.I)
            if m2:
                causes = [c.strip() for c in re.split(r"\s+and\s+", m2.group(1))]
                rules.append((tuple((c, ()) for c in causes), (m2.group(2).strip(), ())))
                continue
            m2 = re.match(r"^(.+?)\s+causes\s+(.+?)$", piece, re.I)
            if m2:
                rules.append((((m2.group(1).strip(), ()),), (m2.group(2).strip(), ())))
                continue
            m2 = re.match(r"^(.+?)\s+occurs$", piece, re.I)
            if m2:
                facts.append((m2.group(1).strip(), ()))
        return facts, rules
    m = re.match(r"^If (.+?) then (.+?)\.?$", s, re.I)
    if m:
        body = parse_condition(m.group(1))
        head = parse_condition(m.group(2))
        if body and head:
            return [], [(tuple(body), head[0])]
    m = re.match(r"^All (?:people|persons) who (.+?) are (.+?)\.?$", s, re.I)
    if m:
        body = parse_condition(m.group(1))
        head = parse_condition(m.group(2))
        if body and head:
            return [], [(tuple(body), head[0])]
    m = re.match(r"^(?:All )?(.+?)\s+(?:things|people|persons)\s+are\s+(.+?)\.?$", s, re.I)
    if m:
        props = split_and(m.group(1))
        head = parse_condition(m.group(2))
        if props and head:
            body = tuple((norm(p), ("X",)) for p in props)
            return [], [(body, head[0])]
    m = re.match(r"^All (.+?) are (.+?)\.?$", s, re.I)
    if m:
        body = parse_condition(m.group(1))
        head = parse_condition(m.group(2))
        if body and head:
            return [], [(tuple(body), head[0])]
    a = parse_atom(s)
    return [a] if a else [], []

def parse_kb(text):
    facts, rules = [], []
    for s in re.split(r"(?<=[.!?])\s+", text):
        s = s.strip()
        if not s:
            continue
        f, r = parse_sentence(s)
        facts.extend(f)
        rules.extend(r)
    return facts, rules

In [6]:
def get_sources(rules):
    bodies, heads = set(), set()
    for body, head in rules:
        heads.add(head)
        for b in body:
            bodies.add(b)
    return bodies - heads

def answer(context, query):
    facts, rules = parse_kb(context)
    q = query.strip()
    m = re.match(r"^Would (.+?) occur if not (.+?) instead of (.+?)\?$", q, re.I)
    if m:
        target = (m.group(1).strip(), ())
        blocked = {(m.group(2).strip(), ())}
        srcs = get_sources(rules) - blocked
        return target in forward(srcs, rules, blocked)
    m = re.match(r"^Would (.+?) occur if (.+?) instead of not (.+?)\?$", q, re.I)
    if m:
        target = (m.group(1).strip(), ())
        srcs = get_sources(rules) | {(m.group(2).strip(), ())}
        return target in forward(srcs, rules)
    m = re.match(r"^Would (.+?) occur if (.+?)\?$", q, re.I)
    if m:
        return (m.group(1).strip(), ()) in forward({(m.group(2).strip(), ())}, rules)
    m = re.match(r"^Would (.+?) occur\?$", q, re.I)
    if m:
        return (m.group(1).strip(), ()) in forward(get_sources(rules), rules)
    m = re.match(r"^Does (.+?) cause (.+?)(?: to occur)?\?$", q, re.I)
    if m:
        return (m.group(2).strip(), ()) in forward({(m.group(1).strip(), ())}, rules)
    query_atom = parse_atom(q)
    return query_atom in forward(facts, rules) if query_atom else False

In [7]:
def evaluate_and_save():
    import csv
    import time

    def to_text(v):
        return " ".join(str(x) for x in v) if isinstance(v, list) else str(v)

    def is_true(v):
        if isinstance(v, bool):
            return v
        return str(v).strip().lower() in ("true", "yes", "entailment", "1")

    rows = []
    datasets = ["ruletaker", "counterbench", "folio"]

    for dataset_name in datasets:
        if dataset_name == "ruletaker":
            ds = load_dataset("tasksource/ruletaker", split="train")
        elif dataset_name == "counterbench":
            p = hf_hub_download(repo_id="CounterBench/CounterBench",filename="data_balanced_alpha_V1.json",repo_type="dataset")
            ds = load_dataset("json", data_files=p, split="train")
        else:
            ds = load_dataset("tasksource/folio", split="train")
        cols = ds.column_names
        ctx_col = next((c for c in ["context", "given_info", "premises"] if c in cols),None)
        qry_col = next((c for c in ["question", "statement", "conclusion"] if c in cols),None)
        lbl_col = next((c for c in ["answer", "label", "flag"] if c in cols),None)
        if not all([ctx_col, qry_col, lbl_col]):
            raise ValueError(f"Unexpected columns in {dataset_name}: {cols}")
        correct = 0
        start = time.time()
        for i, ex in enumerate(ds):
            context = to_text(ex[ctx_col])
            query = to_text(ex[qry_col])
            prediction = bool(answer(context, query))
            actual = is_true(ex[lbl_col])
            if prediction == actual:
                correct += 1
            rows.append([f"{dataset_name}_{i}",str(prediction)])
        elapsed = time.time() - start
        accuracy = correct / len(ds) * 100 if len(ds) else 0
        print(f"{dataset_name}: {accuracy:.2f}% "f"({correct}/{len(ds)}), {elapsed:.2f}s")
    with open("submission.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "predicted_label"])
        writer.writerows(rows)
    print("Total predictions:", len(rows))
    print("Columns: id, predicted_label")
    print("File: submission.csv")

evaluate_and_save()

ruletaker: 70.04% (336293/480152), 326.18s
counterbench: 64.50% (645/1000), 0.11s
folio: 61.34% (614/1001), 0.23s
Total predictions: 482153
Columns: id, predicted_label
File: submission.csv
